# The Central Limit Theorem — Continuous Case

Coins showed the CLT for counts; now a **continuous** starting point: call-center waiting times, uniform between 0 and 15 minutes. Averaging $n$ calls gives a new variable $Y_n$, and its distribution morphs — flat at $n{=}1$, a triangle at $n{=}2$, unmistakably bell-shaped by $n{=}4$ or $5$ — while its mean stays at $\mu = 7.5$ and its variance shrinks to $\sigma^2/n$. Standardizing collapses the whole family onto one curve, giving the formal statement: $\frac{\bar{X}_n - \mu}{\sigma/\sqrt{n}} \to N(0, 1)$. It is based on the seventh video of Lesson 5 (Sampling and Point Estimation).

## 1. Introduction

Back to the tech-support line from Week 1: an operator answers any time between 0 and 15 minutes (then the call disconnects), so the waiting time $X$ is **uniform on $[0, 15]$** — constant density, every equal-length interval equally likely. You want the *mean* wait time, so you average the waits of $n$ calls:

$$
Y_n = \frac{X_1 + X_2 + \cdots + X_n}{n}
$$

What can be said about the **distribution** of $Y_n$? Everything, it turns out — and the answer is the central limit theorem in its full form.

After completing this notebook you will be able to:

- Reproduce the shape evolution of $Y_n$: uniform → triangle → bell.
- Derive $E[Y_n] = \mu$ (linearity) and $\mathrm{Var}(Y_n) = \sigma^2/n$ (independence!).
- State the CLT in standardized-average and in sum form, and the $n \approx 30$ rule of thumb.
- Show how skewness of the starting distribution delays the bell.

## 2. Theory

### The shape evolution

Repeat the experiment "average $n$ waits" many, many times and histogram the results:

- $n = 1$: each sample *is* a uniform draw — the histogram is flat (with ups and downs).
- $n = 2$: a **triangle**, almost symmetric around 7.5, the population mean.
- $n = 3$: starting to look bell-shaped, still centered at 7.5.
- $n = 4, 5$: more bell-shaped, and with **lower dispersion**. Fitting the histograms with a KDE (last lesson's tool) gives curves that look more and more like a Gaussian PDF.

### The mean of $Y_n$: linearity

$$
E[Y_n] = E\left[\frac{1}{n}\sum_i X_i\right] = \frac{1}{n}\sum_i E[X_i] = \frac{1}{n} \cdot n \cdot \mu_X = \mu_X
$$

— the average's expectation equals the population mean, for the uniform$(0,15)$: $\mu = 7.5$. Only linearity was used; no independence needed.

### The variance of $Y_n$: independence

$$
\mathrm{Var}(Y_n) = \mathrm{Var}\left(\frac{1}{n}\sum_i X_i\right)
= \frac{1}{n^2}\,\mathrm{Var}\left(\sum_i X_i\right)
= \frac{1}{n^2}\sum_i \mathrm{Var}(X_i)
= \frac{1}{n^2} \cdot n \sigma^2 = \frac{\sigma^2}{n}
$$

Two facts did the work: the constant $\tfrac1n$ comes out of the variance **squared**, and — *because the variables are independent* — the variance of the sum is the sum of the variances (not a general fact!). For the uniform$(0, 15)$: $\sigma^2 = \frac{(15-0)^2}{12} = 18.75$, so $\mathrm{Var}(Y_n) = \frac{18.75}{n}$.

So: **the mean stays the same, the variance gets smaller and smaller** — more calls averaged, less spread, exactly what the narrowing histograms showed. And this holds *no matter what the original population's distribution is*.

### Overlaying the Gaussian

Plot the Gaussian with $Y_n$'s mean and variance over each KDE: not so close for $n = 1$ or 2, almost indistinguishable by $n = 4$–5. When you average a large enough number of variables, the distribution approximately follows a normal distribution.

**Rule of thumb:** the uniform converges unusually fast; in general expect to need $n \approx 30$ — and *more* if the original population is very skewed.

### Standardizing: the formal statement

Since the mean is fixed but the variance depends on $n$, distributions for different $n$ are easiest to compare after **standardizing**. The central limit theorem, formally:

$$
\frac{\bar{X}_n - \mu}{\sigma / \sqrt{n}} \;\xrightarrow{\;n \to \infty\;}\; N(0, 1)
$$

Equivalently, in **sum form** (factor out the $\tfrac1n$ and rearrange):

$$
\frac{\sum_{i=1}^n X_i - n\mu}{\sqrt{n}\,\sigma} \;\xrightarrow{\;n \to \infty\;}\; N(0, 1)
$$

(A bonus of the standardized form, important next week: even when you *don't know* $\mu$ and $\sigma^2$, you know the average's distribution is approximately Gaussian — the basis for confidence intervals.)

## 3. Worked Examples

### Example 1: the variance table

$\sigma^2 = 18.75$ for one call, so:

| $n$ | 1 | 2 | 3 | 4 | 5 | 30 |
|---|---|---|---|---|---|---|
| $\mathrm{Var}(Y_n)$ | 18.75 | 9.375 | 6.25 | 4.6875 | 3.75 | 0.625 |
| $\mathrm{std}(Y_n)$ | 4.33 | 3.06 | 2.50 | 2.17 | 1.94 | 0.79 |

### Example 2: a CLT probability

Average 30 calls: $Y_{30} \approx N(7.5, 0.625)$, $\sigma_{Y} = 0.79$. Probability the average wait lands between 7 and 8 minutes: standardize — $z$-range $\frac{7 - 7.5}{0.79} = -0.63$ to $+0.63$ → about $47\%$. No 30-dimensional integral required.

### Example 3: why $n=2$ gives a triangle

The sum of two independent uniforms is the *convolution* of two rectangles — a triangle (seen before with dice in Lesson 3's sum-of-Gaussians exercises). Each further averaging convolves again, rounding the corners toward the bell: the CLT is convolution smoothing, repeated.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

a, b = 0, 15
mu = (a + b) / 2
sigma2 = (b - a)**2 / 12
print(f"waiting time ~ Uniform(0, 15): mu = {mu}, sigma^2 = {sigma2}")

def normal_pdf(x, m, s):
    return np.exp(-0.5 * ((x - m) / s)**2) / (s * np.sqrt(2 * np.pi))

def kde(x, data, h):
    return np.mean([normal_pdf(x, xi, h) for xi in data], axis=0)

### Mean and variance of $Y_n$: theory vs simulation

In [ ]:
rng = np.random.default_rng(0)

print(f"{'n':>4} {'E[Yn] sim':>10} {'Var(Yn) sim':>12} {'sigma^2/n':>10}")
for n in [1, 2, 3, 4, 5, 30]:
    y = rng.uniform(0, 15, size=(100_000, n)).mean(axis=1)
    print(f"{n:>4} {y.mean():>10.3f} {y.var():>12.3f} {sigma2/n:>10.3f}")
print("\nMean pinned at 7.5; variance = 18.75/n, shrinking exactly as derived.")

### The video's gallery: histogram + KDE (green) + Gaussian (orange)

The full evolution for $n = 1 \dots 5$: watch the green KDE hug the orange Gaussian ever more closely.

In [ ]:
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.6), sharey=True)
grid = np.linspace(0, 15, 400)
for ax, n in zip(axes, [1, 2, 3, 4, 5]):
    y = rng.uniform(0, 15, size=(20_000, n)).mean(axis=1)
    ax.hist(y, bins=40, density=True, color="tab:blue", alpha=0.4)
    h = 1.06 * y.std() * len(y)**(-1/5)
    ax.plot(grid, kde(grid, y[:1500], h), color="tab:green", linewidth=2,
            label="KDE")
    ax.plot(grid, normal_pdf(grid, mu, np.sqrt(sigma2 / n)), color="tab:orange",
            linewidth=2, label="Gaussian")
    ax.set_title(f"n = {n}")
    ax.set_xlabel("average wait (min)")
    ax.grid(True, alpha=0.3)
    if n == 1:
        ax.legend(fontsize=9)
axes[0].set_ylabel("density")
plt.suptitle("Averages of n uniform waits: flat -> triangle -> bell, tightening around 7.5")
plt.tight_layout()
plt.show()

### Standardizing collapses the family onto N(0, 1)

In [ ]:
rng = np.random.default_rng(1)

grid_z = np.linspace(-4, 4, 300)
plt.figure(figsize=(9.5, 5))
for n, color in [(1, "tab:red"), (2, "tab:orange"), (5, "tab:green"),
                 (30, "tab:blue")]:
    y = rng.uniform(0, 15, size=(200_000, n)).mean(axis=1)
    z = (y - mu) / np.sqrt(sigma2 / n)          # standardize with sigma/sqrt(n)
    plt.hist(z, bins=80, density=True, alpha=0.35, color=color,
             label=f"n = {n}")
plt.plot(grid_z, normal_pdf(grid_z, 0, 1), color="black", linewidth=2.5,
         label="standard normal")
plt.title("Standardized averages: $(\\bar{X}_n - \\mu)/(\\sigma/\\sqrt{n})$ -> N(0, 1)")
plt.xlabel("z")
plt.ylabel("density")
plt.xlim(-4, 4)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The sum form says the same thing

In [ ]:
rng = np.random.default_rng(2)

n = 30
x = rng.uniform(0, 15, size=(200_000, n))
sums = x.sum(axis=1)
z_sum = (sums - n * mu) / (np.sqrt(n) * np.sqrt(sigma2))
z_avg = (x.mean(axis=1) - mu) / (np.sqrt(sigma2) / np.sqrt(n))

print(f"max |z_sum - z_avg| over 200,000 experiments: "
      f"{np.abs(z_sum - z_avg).max():.2e}")
print("Identical: dividing the sum's standardization by n top and bottom gives")
print("the average's -- the two CLT statements are one theorem.")

### Skewed populations need more n

The video's warning quantified: a heavily skewed (lognormal) waiting time vs the uniform, both at $n = 5$ and $n = 30$ — the skewed one still leans at $n = 30$.

In [ ]:
rng = np.random.default_rng(3)

fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))
sources = [("uniform(0, 15)", lambda size: rng.uniform(0, 15, size)),
           ("lognormal (skewed)", lambda size: rng.lognormal(1.5, 1.0, size))]

for row, (name, gen) in enumerate(sources):
    for col, n in enumerate([5, 30]):
        y = gen((100_000, n)).mean(axis=1)
        z = (y - y.mean()) / y.std()
        skew = np.mean(z**3)
        axes[row, col].hist(z, bins=80, density=True, alpha=0.6,
                            color="tab:blue" if row == 0 else "tab:orange")
        axes[row, col].plot(grid_z, normal_pdf(grid_z, 0, 1), color="black",
                            linewidth=2)
        axes[row, col].set_title(f"{name}, n = {n}  (skew {skew:+.2f})", fontsize=10)
        axes[row, col].set_xlim(-4, 4)
        axes[row, col].grid(True, alpha=0.3)
plt.suptitle("The CLT always applies -- skewed starts just take longer to kick in")
plt.tight_layout()
plt.show()

## 5. Visualizations

### The narrowing bell: $Y_n$ for growing n on one axis

In [ ]:
rng = np.random.default_rng(5)

plt.figure(figsize=(10, 5))
grid = np.linspace(0, 15, 500)
for n, color in [(1, "tab:red"), (3, "tab:orange"), (10, "tab:green"),
                 (30, "tab:blue"), (100, "tab:purple")]:
    plt.plot(grid, normal_pdf(grid, mu, np.sqrt(sigma2 / n)), color=color,
             linewidth=2, label=f"n = {n}  (std {np.sqrt(sigma2/n):.2f})")
plt.axvline(mu, color="black", linestyle="--", linewidth=1)
plt.title("The approximating Gaussians: same center 7.5, width $\\sigma/\\sqrt{n}$")
plt.xlabel("average wait (minutes)")
plt.ylabel("density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### CLT at work on a real question

$P(7 \le Y_{30} \le 8)$: the simulation histogram, the Gaussian, and the shaded probability agreeing.

In [ ]:
rng = np.random.default_rng(6)

y30 = rng.uniform(0, 15, size=(300_000, 30)).mean(axis=1)
p_sim = np.mean((y30 >= 7) & (y30 <= 8))

s30 = np.sqrt(sigma2 / 30)
grid = np.linspace(5, 10, 500)
pdf30 = normal_pdf(grid, mu, s30)
mask = (grid >= 7) & (grid <= 8)
p_clt = np.trapezoid(pdf30[mask], grid[mask])

plt.figure(figsize=(9, 4.5))
plt.hist(y30, bins=80, density=True, alpha=0.5, color="tab:blue",
         label="simulated $Y_{30}$")
plt.plot(grid, pdf30, color="tab:orange", linewidth=2, label="CLT Gaussian")
plt.fill_between(grid[mask], pdf30[mask], color="tab:orange", alpha=0.4)
plt.title(f"P(7 <= Y_30 <= 8): simulation {p_sim:.3f} vs CLT {p_clt:.3f}")
plt.xlabel("average wait of 30 calls (minutes)")
plt.ylabel("density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Derive the uniform$(0,15)$ variance $18.75$ from the general formula $\frac{(b-a)^2}{12}$, and verify it numerically with `np.trapezoid` (as in the Lesson 3 variance notebook).

**Exercise 2.** For $n = 100$ calls: what are the mean, variance, and standard deviation of $Y_{100}$? Between which two values does the average wait land ~95% of the time?

### Intermediate

**Exercise 3.** How many calls must you average so that the standard deviation of $Y_n$ is below half a minute? Solve $\sigma/\sqrt{n} < 0.5$ and confirm by simulation.

**Exercise 4.** The $n = 2$ triangle, exactly: derive (or verify numerically by dense simulation) that the density of $Y_2$ is triangular on $[0, 15]$ peaking at 7.5, and check that the CLT Gaussian at $n=2$ *underestimates* the peak — the bell hasn't formed yet.

### Advanced

**Exercise 5.** Design the stress test: find a starting distribution where even $n = 30$ leaves visible skew in the standardized average (try lognormal with $\sigma = 2$), and estimate how large $n$ must get for |skew| < 0.1. Relate to the skewness-of-averages formula skew$/\sqrt{n}$ from the Lesson 3 skewness notebook.

**Exercise 6.** The bridge to next week: suppose you *don't know* $\mu$ and $\sigma$ of the waiting-time distribution. Simulate one sample of $n = 50$ calls, standardize its mean using the *sample* estimates ($\bar{x}$, $s$), and repeat 10,000 times: how close is the histogram of $\frac{\bar{x} - \mu}{s/\sqrt{n}}$ to N(0,1)? (This object — the t-statistic — is what confidence intervals are built on.)

## 7. Solutions

In [ ]:
# Exercise 1: the uniform variance
x = np.linspace(0, 15, 100_001)
f = np.full_like(x, 1 / 15)
var_num = np.trapezoid((x - 7.5)**2 * f, x)
print(f"Exercise 1: (b-a)^2/12 = 225/12 = {225/12}")
print(f"numerical integral: {var_num:.4f}")

In [ ]:
# Exercise 2: Y_100
var100 = sigma2 / 100
s100 = np.sqrt(var100)
print(f"Exercise 2: E = {mu}, Var = {var100:.4f}, std = {s100:.3f}")
print(f"95% band: mu +- 2 std = [{mu - 2*s100:.2f}, {mu + 2*s100:.2f}] minutes")

In [ ]:
# Exercise 3: calls needed for std < 0.5
rng = np.random.default_rng(0)

n_needed = int(np.ceil(sigma2 / 0.5**2))
print(f"Exercise 3: sigma/sqrt(n) < 0.5  =>  n > sigma^2/0.25 = {sigma2/0.25}")
print(f"so n = {n_needed}")
y = rng.uniform(0, 15, size=(100_000, n_needed)).mean(axis=1)
print(f"simulated std at n = {n_needed}: {y.std():.3f}")

In [ ]:
# Exercise 4: the exact triangle vs the premature bell
rng = np.random.default_rng(1)

y2 = rng.uniform(0, 15, size=(2_000_000, 2)).mean(axis=1)
grid = np.linspace(0, 15, 400)
triangle = np.where(grid <= 7.5, grid / 7.5**2 * (grid > 0),
                    (15 - grid) / 7.5**2)  # peak 1/7.5 at 7.5

plt.figure(figsize=(9, 4.5))
plt.hist(y2, bins=100, density=True, alpha=0.5, color="tab:blue",
         label="simulated $Y_2$")
plt.plot(grid, triangle, color="tab:green", linewidth=2, label="exact triangle")
plt.plot(grid, normal_pdf(grid, mu, np.sqrt(sigma2 / 2)), color="tab:orange",
         linewidth=2, label="CLT Gaussian (n = 2)")
plt.title("n = 2: the true density is a triangle; the Gaussian is not there yet")
plt.xlabel("average of two waits")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"Exercise 4: triangle peak = 1/7.5 = {1/7.5:.4f}; "
      f"Gaussian at 7.5 = {normal_pdf(7.5, mu, np.sqrt(sigma2/2)):.4f}")
print("The bell underestimates the sharp peak -- averaging two values only")
print("convolves two rectangles once; the corners need more convolutions.")

In [ ]:
# Exercise 5: a stress test for the n = 30 rule
rng = np.random.default_rng(2)

print("Exercise 5: lognormal(0, 2) averages, skew vs n:")
base = rng.lognormal(0, 2, size=(4000, 4096))
single_skew = None
for n in [30, 256, 1024, 4096]:
    y = base[:, :n].mean(axis=1)
    z = (y - y.mean()) / y.std()
    sk = np.mean(z**3)
    if n == 30:
        single_skew = sk
    print(f"  n = {n:>5}: skew of the average = {sk:+.2f}")
print("Far from 0.1 even at n = 4096: with extreme starting skew (lognormal")
print("sigma=2 has skew in the hundreds), skew/sqrt(n) shrinks agonizingly")
print("slowly -- the n ~ 30 rule assumes well-behaved data.")

In [ ]:
# Exercise 6: the t-statistic preview
rng = np.random.default_rng(3)

n = 50
samples = rng.uniform(0, 15, size=(10_000, n))
xbars = samples.mean(axis=1)
ss = samples.std(axis=1, ddof=1)
t = (xbars - mu) / (ss / np.sqrt(n))

grid_z = np.linspace(-4, 4, 300)
plt.figure(figsize=(9, 4.5))
plt.hist(t, bins=80, density=True, alpha=0.55, color="tab:purple",
         label="$(\\bar{x} - \\mu)/(s/\\sqrt{n})$ with SAMPLE s")
plt.plot(grid_z, normal_pdf(grid_z, 0, 1), color="black", linewidth=2,
         label="N(0, 1)")
plt.title("Standardizing with estimated s: nearly standard normal at n = 50")
plt.xlabel("t")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"Exercise 6: std of the t-values: {t.std():.3f} (slightly above 1 --")
print("the extra wobble from estimating s; at small n this becomes the")
print("t-distribution, the workhorse of next week's confidence intervals.)")

## 8. Conclusion

- Averaging $n$ uniform$(0,15)$ waiting times produces $Y_n$ whose distribution **morphs**: flat ($n{=}1$) → triangle ($n{=}2$) → bell ($n \ge 4$), always centered at $\mu = 7.5$ and narrowing.
- The two parameter facts, derived: $E[Y_n] = \mu$ (pure linearity) and $\mathrm{Var}(Y_n) = \frac{\sigma^2}{n}$ ($\tfrac1n$ exits squared; variances of a sum add **only by independence**) — for the uniform, $\frac{18.75}{n}$. Mean stays, spread shrinks: more averaging, closer to $\mu$.
- **The central limit theorem:** $\frac{\bar{X}_n - \mu}{\sigma/\sqrt{n}} \to N(0,1)$ as $n \to \infty$ — equivalently, in sum form, $\frac{\sum X_i - n\mu}{\sqrt{n}\sigma} \to N(0,1)$ — *whatever* the starting distribution.
- Practical guidance: $n \approx 30$ usually suffices; symmetric well-behaved starts (like the uniform) converge much sooner, heavily skewed ones later — the CLT still applies, it just takes longer to kick in.
- Standardization is also the door to next week: even with unknown $\mu, \sigma$, the average is approximately Gaussian — the foundation of confidence intervals and hypothesis tests.

This completes the Lesson 5 core: populations and samples, point estimates ($\bar x$, $\hat p$, $s^2$), the law of large numbers, and the central limit theorem in both discrete and continuous garb.